# Topic Coverage – UMAP Projection
BERTopic pipeline: embeddings → UMAP → HDBSCAN → GPT label mapping → coverage report → 2D plot.

In [ ]:
# ================================
# IMPORTS
# ================================
import re
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import umap
import openai
from umap import UMAP
from hdbscan import HDBSCAN
from scipy.stats import gaussian_kde
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer

In [ ]:
# ================================
# CONFIG
# ================================
USER = "2714"
APIKEY = ""

text_path = rf"C:\Users\Franco\Desktop\Pubblicazione\17\topic_extract\{USER}_Topic_text.txt"
traj_path = rf"C:\Users\Franco\Desktop\Pubblicazione\17\topic_extract\{USER}_Topic_traj.txt"
base_path = rf"C:\Users\Franco\Desktop\Pubblicazione\17\topic_extract\{USER}_Topic_base.txt"
data_path = rf"C:\Users\Franco\Desktop\Pubblicazione\04\utenti_data\{USER}_data.csv"
match_dir = rf"C:\Users\Franco\Desktop\Pubblicazione\21\user"

name = "base"  # name for output files

In [ ]:
# ================================
# LOAD DATA
# ================================
# Replace paths above with your actual file locations
with open(text_path, "r", encoding="utf-8") as f:
    text = json.load(f)  # load the JSON as a Python dictionary or list
with open(traj_path, "r", encoding="utf-8") as f:
    traj = json.load(f)  # load the JSON as a Python dictionary or list
with open(base_path, "r", encoding="utf-8") as f:
    base = json.load(f)  # load the JSON as a Python dictionary or list

df = pd.read_csv(data_path)

l_traj = traj['positivetopics'] + traj['negativetopics'] + traj['neutraltopics']
l_text = text['positivetopics'] + text['negativetopics'] + text['neutraltopics']
l_base = base['positivetopics'] + base['negativetopics'] + base['neutraltopics']

gt_list_str = "\n".join([f"- {gt}" for gt in df['Text']])
df['Text']

In [ ]:
# ================================
# 1. DATA
# ================================
raw_posts = df["Text"].dropna().tolist()
print(f"Analysis on {len(raw_posts)} total posts.")

df_posts = pd.DataFrame({"post_content": raw_posts})

In [ ]:
# ================================
# 2. EMBEDDING MODEL
# ================================
print("Computing embeddings (all-mpnet-base-v2)...")
embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

post_embeddings = embedding_model.encode(
    df_posts["post_content"].tolist(), show_progress_bar=True
)

In [ ]:
# ================================
# 3. BERTopic: UMAP -> HDBSCAN -> c-TF-IDF
# ================================
print("Building BERTopic model...")

umap_model = UMAP(
    n_neighbors=50,
    n_components=5,
    metric="cosine",
    min_dist=0.0,
    random_state=42,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

vectorizer_model = CountVectorizer()
ctfidf_model = ClassTfidfTransformer()

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics=16,
    verbose=True,
)

topics, probs = topic_model.fit_transform(df_posts["post_content"].tolist())
df_posts["topic_id"] = topics
unique_topics = sorted(t for t in np.unique(topics) if t != -1)

topic_info = topic_model.get_topic_info()
topic_info

In [ ]:
# ================================
# 4. GLOBAL 1-TO-1 MAPPING ON EXTRACTED TOPICS VIA GPT (OpenAI)
# ================================
print("Extracting keywords and performing global 1-to-1 mapping...")

client = openai.OpenAI(api_key=APIKEY)

# Build a dictionary with Topic IDs and their related keywords (ignoring outlier -1)
bertopic_topics = {}
for idx, row in topic_info.iterrows():
    t = row['Topic']
    if t == -1:
        continue
    # Extract the top 10 keywords for each topic
    keywords = ", ".join([word for word, score in topic_model.get_topic(t)[:10]])
    bertopic_topics[str(t)] = keywords

# Build the GLOBAL prompt using the l_text list (assumed as default)
global_prompt = f"""
I have extracted a set of topics using BERTopic. Each topic is identified by an ID and described by its top keywords:
{json.dumps(bertopic_topics, indent=2)}

I also have a predefined list of target labels:
{l_text}

Your task is to act as a matcher and assign exactly ONE target label to EACH extracted topic based on semantic similarity.

CRITICAL RULES:
1. STRICT 1-TO-1 MAPPING: You MUST NOT use the same label twice. Every assignment must be unique.
2. DO NOT invent new labels. You must exclusively use the labels provided in the predefined list.
3. Every topic label MUST have logical and grammatical sense.
4. Return ONLY a valid JSON object where the keys are the exact Topic IDs (as strings, e.g., "0", "1", "2") and the values are the assigned predefined labels. Do not include markdown formatting like ```json or any other text.
"""

# Single GPT-4o call to map everything at once in JSON format
response = client.chat.completions.create(
    model="gpt-4o",
    response_format={"type": "json_object"},
    messages=[{"role": "user", "content": global_prompt}],
    temperature=0.0
)

# Parsing the JSON response
mapped_topics_json = json.loads(response.choices[0].message.content)
mapped_topics_json

In [ ]:
# ================================
# 5. VALIDATION & LABEL ASSIGNMENT
# ================================
# Verify that EVERY assigned label is actually in l_text.


valid_labels = set(l_text)
used_labels = set()
invalid_topics = []  # topics with invalid labels

# First pass: identify valid and invalid labels
for t_str, assigned_label in mapped_topics_json.items():
    if assigned_label in valid_labels and assigned_label not in used_labels:
        used_labels.add(assigned_label)
    else:
        invalid_topics.append(t_str)

if invalid_topics:
    unused_labels = [lbl for lbl in l_text if lbl not in used_labels]
   
    if unused_labels:
        # Compute embeddings of unused labels
        unused_embeddings = embedding_model.encode(unused_labels)

        for t_str in invalid_topics:
            # Embedding of topic keywords
            topic_keywords = bertopic_topics[t_str]
            topic_emb = embedding_model.encode([topic_keywords])

            # Find the most similar unused label
            similarities = cosine_similarity(topic_emb, unused_embeddings)[0]
            best_idx = int(np.argmax(similarities))
            best_label = unused_labels[best_idx]

            mapped_topics_json[t_str] = best_label
            used_labels.add(best_label)

            # Remove the just-used label from the list
            unused_labels.pop(best_idx)
            unused_embeddings = np.delete(unused_embeddings, best_idx, axis=0)

            print(f"  Topic {t_str}: '{bertopic_topics[t_str][:50]}...' -> {best_label}")


# Update labels for use in the rest of the script
gpt_labels = {}
new_representations = {}

for t_str, assigned_label in mapped_topics_json.items():
    t_int = int(t_str)
    display_t = t_int + 1

    gpt_labels[display_t] = f"{display_t}: {assigned_label}"
    new_representations[t_int] = [assigned_label]

# Also add topic -1 (outlier)
new_representations[-1] = ["Outlier"]

# Visually update the BERTopic model
topic_model.set_topic_labels(new_representations)

print("Global 1-to-1 label assignment completed successfully!\n")
for k, v in gpt_labels.items():
    print(f"Topic {k} -> {v}")

In [ ]:
# ================================
# 6. UMAP 2D FOR VISUALIZATION
# ================================
print("\nRunning UMAP 2D for visualization...")
reducer = umap.UMAP(
    n_neighbors=15,
    n_components=2,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

coords = reducer.fit_transform(post_embeddings)
df_posts["x"] = coords[:, 0]
df_posts["y"] = coords[:, 1]

print("UMAP 2D projection completed.")
df_posts.head()

In [ ]:
# ================================
# 7. CHECKING TOPIC PRESENCE IN REPORTS (from pre-existing files)
# ================================
print("\n--- Reading topic coverage from match/miss files ---")

base_match_path = os.path.join(match_dir, f"{USER}_match_Topic_base.txt")
traj_match_path = os.path.join(match_dir, f"{USER}_match_Topic_traj.txt")

def parse_matched_topics(filepath):
    """Reads a match/miss file and returns the list of matched topics."""
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    # Extract only the "Match List" section (between "Match List" and "Miss List")
    match_section = ""
    if "Match List" in content and "Miss List" in content:
        match_section = content.split("Match List")[1].split("Miss List")[0]
    elif "Match List" in content:
        match_section = content.split("Match List")[1]

    # Find all topics in markdown bold: **topic name**
    matched = re.findall(r'\*\*(.+?)\*\*', match_section)
    # Strip trailing punctuation (e.g. ":" or ".") and whitespace
    matched = [m.strip().rstrip(':.') for m in matched]
    return matched

def build_presence_dict(matched_topics_list, labels_dict):
    """Builds a {topic_id_str: True/False} dictionary by comparing
    matched topics from the file with labels assigned by GPT."""
    presence = {}
    for topic_id, label_str in labels_dict.items():
        # label_str is like "1: Self-harm urges and behaviors", extract only the name
        label_name = label_str.split(": ", 1)[1] if ": " in label_str else label_str

        # Check if this topic is in the matched list
        # Use bidirectional comparison to handle minor differences
        label_lower = label_name.lower()
        is_covered = any(
            label_lower == m.lower() or
            label_lower in m.lower() or
            m.lower() in label_lower
            for m in matched_topics_list
        )
        presence[str(topic_id)] = is_covered
    return presence

base_matched_topics = parse_matched_topics(base_match_path)
traj_matched_topics = parse_matched_topics(traj_match_path)

print(f"  Base: {len(base_matched_topics)} matched topics found in file")
print(f"  Traj: {len(traj_matched_topics)} matched topics found in file")

base_presence = build_presence_dict(base_matched_topics, gpt_labels)
traj_presence = build_presence_dict(traj_matched_topics, gpt_labels)

print("\nReading completed!")

In [ ]:
# ================================
# 8. COVERAGE COUNT AND PRINT
# ================================
total_topics = len(gpt_labels)
num_base_covered = sum(1 for v in base_presence.values() if v is True)
num_traj_covered = sum(1 for v in traj_presence.values() if v is True)

print("\n=== TOPIC COVERAGE RESULTS ===")
print(f"Base Report:       {num_base_covered} topics covered out of {total_topics} total.")
print(f"Trajectory Report: {num_traj_covered} topics covered out of {total_topics} total.")
print("==============================\n")

In [ ]:
# ================================
# 9. PLOT: colors = BERTopic topics, labels = GPT Labels + Shapes for Reports
# ================================
plt.figure(figsize=(14, 10))

plot_topics = unique_topics
cmap = plt.cm.get_cmap("tab20", len(plot_topics))
topic_to_color = {t: cmap(i) for i, t in enumerate(plot_topics)}

for t in plot_topics:
    subset = df_posts[df_posts["topic_id"] == t]
    if not subset.empty:
        # Start from 1 for display
        display_t = t + 1
        gpt_label = gpt_labels.get(display_t, f"Topic {display_t}")

        plt.scatter(
            subset["x"], subset["y"],
            s=40, alpha=0.7, color=topic_to_color[t],
            label=gpt_label
        )

        cx, cy = subset["x"].mean(), subset["y"].mean()
        str_t = str(display_t)

        # Find the densest point of the cluster using KDE
        points = np.column_stack([subset["x"].values, subset["y"].values])
        if len(points) > 3:
            kde = gaussian_kde(points.T)
            densities = kde(points.T)
            dense_idx = int(np.argmax(densities))
            dx, dy = points[dense_idx]
        else:
            dx, dy = cx, cy

        # Small offset to avoid overlapping diamond and star markers
        small_off = 0.15

        # 1. Diamond for the "Base" report (Red) — at densest point, slightly left
        if base_presence.get(str_t, False):
            plt.scatter(dx - small_off, dy, marker='D', s=120,
                        color='red', zorder=8)

        # 2. Star for the "Trajectory" report (Blue) — at densest point, slightly right
        if traj_presence.get(str_t, False):
            plt.scatter(dx + small_off, dy, marker='*', s=200,
                        color='blue', zorder=9)

        # 3. Central text with the ID (starting from 1)
        plt.text(
            cx, cy, str_t,
            fontsize=11, fontweight="bold",
            bbox=dict(boxstyle="circle", fc="white", ec=topic_to_color[t], alpha=0.9),
            ha='center', va='center', zorder=10
        )

plt.title("Clustering of Posts", fontsize=16)
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")

# Legend icons
base_marker = mlines.Line2D([], [], color='red', marker='D', linestyle='None', markersize=8, label='Report Base')
traj_marker = mlines.Line2D([], [], color='blue', marker='*', linestyle='None', markersize=12, label='Report Traj')

handles, labels = plt.gca().get_legend_handles_labels()
handles.extend([base_marker, traj_marker])
labels.extend(['Included in the Base Report', 'Included in the Trajectory Report'])
plt.legend(
    handles=handles, labels=labels,
    bbox_to_anchor=(1.02, 1.0),
    loc="upper left",
    title="Topic",
    fontsize=7,
    framealpha=0.95,
    ncol=1
)

plt.grid(True, alpha=0.3)
plt.margins(0.05)
plt.tight_layout()

# === VECTOR SAVE ===
plt.savefig(f"bertopic_umap_{USER}_reports.pdf", format="pdf", bbox_inches="tight")
plt.show()
print(f"Plot saved to: bertopic_umap_{USER}_reports.pdf")